In [ ]:
# ==========================================
# SECTION 1: A/B TEST 1 - NEW ONBOARDING FLOW
# ==========================================
"""
MARKDOWN CELL:
### Experiment: New Onboarding Flow
- **Hypothesis (H1):** A simplified onboarding flow will increase the onboarding completion rate compared to the current flow.
- **Null Hypothesis (H0):** There is no difference in completion rates between the two flows.
- **Primary Metric:** Onboarding Completion Rate.
- **Secondary Metric:** Day-7 Retention.
"""

import pandas as pd
import numpy as np
import scipy.stats as stats
import math

# Load Data
users = pd.read_csv('../data/users.csv')
events = pd.read_csv('../data/events.csv')
ab_tests = pd.read_csv('../data/ab_tests.csv')

# Filter for Experiment 1
exp1 = ab_tests[ab_tests['experiment_name'] == 'new_onboarding_flow'].copy()

# Find users who completed onboarding
completed_users = events[events['event_type'] == 'onboarding_complete']['user_id'].unique()
exp1['converted'] = exp1['user_id'].isin(completed_users).astype(int)

# Calculate stats
control_results = exp1[exp1['variant'] == 'control']['converted']
treatment_results = exp1[exp1['variant'] == 'treatment']['converted']

n_con = len(control_results)
n_treat = len(treatment_results)
success_con = control_results.sum()
success_treat = treatment_results.sum()

rate_con = success_con / n_con
rate_treat = success_treat / n_treat

print(f"--- Onboarding Flow Results ---")
print(f"Control: {n_con} users, Conversion Rate: {rate_con:.2%}")
print(f"Treatment: {n_treat} users, Conversion Rate: {rate_treat:.2%}")
print(f"Absolute Lift: {(rate_treat - rate_con):.2%}")

# Chi-Squared Test
contingency_table = [[success_con, n_con - success_con], 
                     [success_treat, n_treat - success_treat]]
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"P-Value: {p_value:.5f}")

# Confidence Interval (95%)
z_score = 1.96 
se_diff = math.sqrt((rate_con * (1 - rate_con) / n_con) + (rate_treat * (1 - rate_treat) / n_treat))
margin_of_error = z_score * se_diff
ci_lower = (rate_treat - rate_con) - margin_of_error
ci_upper = (rate_treat - rate_con) + margin_of_error
print(f"95% Confidence Interval for the Lift: [{ci_lower:.2%}, {ci_upper:.2%}]")

"""
MARKDOWN CELL:
**DECISION: SHIP IT.**
- **Statistical Significance:** The p-value is extremely low (< 0.05), allowing us to reject the null hypothesis.
- **Practical Significance:** The confidence interval shows a massive positive lift.
"""

# ==========================================
# SECTION 2: A/B TEST 2 - PRICING PAGE REDESIGN
# ==========================================
"""
MARKDOWN CELL:
### Experiment: Pricing Page Redesign
- **Hypothesis (H1):** Removing the "Basic" plan visual emphasis will push more users to the "Pro" plan, increasing overall upgrade rate and MRR.
- **Primary Metric:** Upgrade Conversion Rate (Free to Paid).
- **Secondary Metric:** Expected Revenue Impact.
"""

# Filter for Experiment 2
exp2 = ab_tests[ab_tests['experiment_name'] == 'pricing_page_redesign'].copy()

# Find users who paid
paid_users = events[events['event_type'] == 'payment_success']['user_id'].unique()
exp2['converted'] = exp2['user_id'].isin(paid_users).astype(int)

# Calculate stats
control_results_2 = exp2[exp2['variant'] == 'control']['converted']
treatment_results_2 = exp2[exp2['variant'] == 'treatment']['converted']

n_con_2 = len(control_results_2)
n_treat_2 = len(treatment_results_2)
success_con_2 = control_results_2.sum()
success_treat_2 = treatment_results_2.sum()

rate_con_2 = success_con_2 / n_con_2
rate_treat_2 = success_treat_2 / n_treat_2

print(f"\n--- Pricing Page Results ---")
print(f"Control: {n_con_2} users, Conversion Rate: {rate_con_2:.2%}")
print(f"Treatment: {n_treat_2} users, Conversion Rate: {rate_treat_2:.2%}")

chi2_2, p_value_2, dof_2, expected_2 = stats.chi2_contingency(
    [[success_con_2, n_con_2 - success_con_2], [success_treat_2, n_treat_2 - success_treat_2]]
)
print(f"P-Value: {p_value_2:.5f}")

# Revenue Impact Estimation
expected_lift = rate_treat_2 - rate_con_2
annual_visitors = 100000
additional_conversions = annual_visitors * expected_lift
expected_mrr_impact = additional_conversions * 30 / 12

print(f"Expected MRR Impact if rolled out to 100k users/year: ${expected_mrr_impact:,.2f}/month")